In [1]:
# this notebook will normalize(scale)/ clean the data
# while keeping key level as feature
# revert file : Cmd+Shift+P 
import pandas as pd
import json
from pathlib import Path

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

folder_path = find_project_root() / "data" / "mlData"

symbol = "BTCUSDT"
tf = "5m"

# 65 MB with almost 50 cols, LOL
folder_path = find_project_root() / "data" / "mlData" 
src_path = folder_path / "augmented" / f"{symbol}-{tf}-v10-augmented.jsonl"

# Read JSONL file - keep timestamp as raw number
df = pd.read_json(src_path, lines=True, convert_dates=False)

# extra col : experimenting data
# df["DELTA_2"] = df["DELTA_1"].rolling(2).sum()

print(df.columns)
df.head()

Index(['timestamp', 'label', '15STR_confirmed', '45STR_confirmed',
       'barsSince15STR', 'barsSince45STR', 'ROC_3', 'ROC_5', 'ROC_10', 'MOM_3',
       'RETURNS_1', 'DELTA_1', 'BUY_RATIO', 'DELTA_3', 'VOL_SPIKE',
       'DELTA_DIV', 'ATR_5', 'ATR_14', 'ATR_RATIO', 'ATR_NORM_ROC',
       'RANGE_RATIO', 'RSI_5', 'RSI_14', 'RSI_SLOPE', 'STOCH_K', 'CCI_5',
       'DIST_HIGH_5', 'DIST_LOW_5', 'DIST_HIGH_10', 'DIST_LOW_10', 'RANGE_POS',
       'DIST_HIGH_15STR', 'DIST_LOW_15STR', 'DIST_HIGH_45STR',
       'DIST_LOW_45STR', 'RANGE_15STR', 'RANGE_45STR'],
      dtype='object')


,timestamp,label,15STR_confirmed,45STR_confirmed,barsSince15STR,barsSince45STR,ROC_3,ROC_5,ROC_10,MOM_3,...,DIST_LOW_5,DIST_HIGH_10,DIST_LOW_10,RANGE_POS,DIST_HIGH_15STR,DIST_LOW_15STR,DIST_HIGH_45STR,DIST_LOW_45STR,RANGE_15STR,RANGE_45STR
0,1703910600000,NaN,-1,0,0,0,0.000000,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,NaN,0.339811,0.984269,0.339811,0.984269,0.743361,0.743361
1,1703910900000,NaN,0,0,1,1,0.000371,0.000371,0.000371,0.421754,...,NaN,NaN,NaN,NaN,-0.081943,1.406022,-0.081943,1.406022,1.061887,1.061887
2,1703911200000,NaN,0,0,2,2,0.000418,0.000418,0.000418,0.475186,...,NaN,NaN,NaN,NaN,0.138438,1.459455,0.138438,1.459455,0.913362,0.913362
3,1703911500000,NaN,0,0,3,3,-0.000247,-0.000247,-0.000247,-0.281099,...,NaN,NaN,NaN,NaN,1.372971,0.703170,1.372971,0.703170,0.338691,0.338691
4,1703911800000,NaN,0,0,4,4,-0.001334,-0.000963,-0.000963,-1.515738,...,NaN,NaN,NaN,NaN,2.185855,-0.109715,2.185855,-0.109715,-0.052846,-0.052846


# Analyse Data
- Drop warm-up rows
- check Infinity, NaN, None
- verify data's quality. Non-staionary, local memory, etc.

In [2]:
# removing warmup nan and check label imbalance
import numpy as np
# Replace inf with NaN 
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# list feature columns
# Drop RSI_5 from VIF test
# Drop '15STR_confirmed','45STR_confirmed','barsSince15STR','barsSince45STR' from spearman test
group_I = ['ROC_3', 'ROC_5', 'ROC_10', 'MOM_3', 'RETURNS_1']
group_J = ['ATR_5', 'ATR_14', 'ATR_RATIO', 'ATR_NORM_ROC', 'RANGE_RATIO']
group_K = ['RSI_5', 'RSI_14', 'RSI_SLOPE', 'STOCH_K', 'CCI_5']
group_L = ['DELTA_1', 'BUY_RATIO', 'DELTA_3', 'VOL_SPIKE', 'DELTA_DIV']
group_M = ['DIST_HIGH_5', 'DIST_LOW_5', 'DIST_HIGH_10', 'DIST_LOW_10', 'RANGE_POS']
group_N = [
    '15STR_confirmed', '45STR_confirmed',
    'barsSince15STR',  'barsSince45STR',
    'DIST_HIGH_15STR', 'DIST_LOW_15STR',
    'DIST_HIGH_45STR', 'DIST_LOW_45STR',
    'RANGE_15STR',     'RANGE_45STR',
]

feature_cols = [*group_I, *group_J, *group_K, *group_L, *group_M, *group_N]

# make sure that we drop only warm-up rows
df_clean = df.dropna(subset=feature_cols).reset_index(drop=True)

dropped = len(df) - len(df_clean)
print(f"Rows before : {len(df):,}")
print(f"Rows dropped: {dropped:,}  (warmup NaN from rolling windows)")
print(f"Rows after cleaned  : {len(df_clean):,}")

Rows before : 229,771
Rows dropped: 13  (warmup NaN from rolling windows)
Rows after cleaned  : 229,758


In [3]:
# Trim start-up and tail NaN rows from label (only contiguous edges)
# also make sure there was not supposed to had any NaN left in any rows or columns

label = df_clean['label']

# Find first and last valid label index
first_valid_idx = label.first_valid_index()
last_valid_idx  = label.last_valid_index()

df_clean = df_clean.loc[first_valid_idx:last_valid_idx].reset_index(drop=True)

print(f"Label NaN trimmed — head up to index {first_valid_idx}, tail after index {last_valid_idx}")
print(f"Rows after trim: {len(df_clean):,}")

# Chronological order must still hold
assert df_clean['timestamp'].is_monotonic_increasing, "Timestamp order broken after trim!"

# Assert no NaN remains anywhere — only contiguous edge NaN in label were expected
remaining_nan = df_clean.isnull().sum()
remaining_nan = remaining_nan[remaining_nan > 0]
assert len(remaining_nan) == 0, f"Unexpected NaN still present:\n{remaining_nan.to_string()}"

print("Chronological order: OK")
print("All columns clean: no NaN remaining.")

# train all 3 labels
trainable = df_clean
n_up   = (trainable['label'] ==  1).sum()
n_down = (trainable['label'] == -1).sum()
total  = len(trainable)

print(f"\nLabel balance (trainable only):")
print(f"  Up   ( 1) : {n_up:,}  ({n_up   / total * 100:.1f}%)")
print(f"  Down (-1) : {n_down:,}  ({n_down / total * 100:.1f}%)")
print(f"  Timeout   : {(df_clean['label'] == 0).sum():,}")
print(f"  Total     : {total:,}  ({total / len(df_clean) * 100:.1f}% of clean rows)")


Label NaN trimmed — head up to index 28, tail after index 229757
Rows after trim: 229,730
Chronological order: OK
All columns clean: no NaN remaining.

Label balance (trainable only):
  Up   ( 1) : 76,565  (33.3%)
  Down (-1) : 78,157  (34.0%)
  Timeout   : 75,008
  Total     : 229,730  (100.0% of clean rows)


In [4]:
# Nan and Infinity re-check
# import numpy as np

# 1. Feature columns must be clean (no NaN/Inf from warm-up)
assert df_clean[feature_cols].isnull().sum().sum() == 0, "NaN leaked through on feature columns!"
assert not np.isinf(df_clean[feature_cols].values).any(), "Inf leaked through on feature columns!"

# 2. Show any remaining NaN per column (expected only in 'label' from source data)
nan_counts = df_clean.isnull().sum()
nan_remaining = nan_counts[nan_counts > 0]
if len(nan_remaining) > 0:
    print("NaN remaining (source data NaN, NOT warm-up):")
    print(nan_remaining.to_string())
else:
    print("No NaN remaining in any column.")

# 3. Verify chronological order is preserved
assert df_clean['timestamp'].is_monotonic_increasing, "Timestamp order broken!"
print(f"\nChronological order: OK (timestamps monotonically increasing)")
print(f"Feature columns: clean (no NaN, no Inf)")


No NaN remaining in any column.

Chronological order: OK (timestamps monotonically increasing)
Feature columns: clean (no NaN, no Inf)


In [5]:
print(df_clean.shape)
print(df_clean.columns)

(229730, 37)
Index(['timestamp', 'label', '15STR_confirmed', '45STR_confirmed',
       'barsSince15STR', 'barsSince45STR', 'ROC_3', 'ROC_5', 'ROC_10', 'MOM_3',
       'RETURNS_1', 'DELTA_1', 'BUY_RATIO', 'DELTA_3', 'VOL_SPIKE',
       'DELTA_DIV', 'ATR_5', 'ATR_14', 'ATR_RATIO', 'ATR_NORM_ROC',
       'RANGE_RATIO', 'RSI_5', 'RSI_14', 'RSI_SLOPE', 'STOCH_K', 'CCI_5',
       'DIST_HIGH_5', 'DIST_LOW_5', 'DIST_HIGH_10', 'DIST_LOW_10', 'RANGE_POS',
       'DIST_HIGH_15STR', 'DIST_LOW_15STR', 'DIST_HIGH_45STR',
       'DIST_LOW_45STR', 'RANGE_15STR', 'RANGE_45STR'],
      dtype='object')


# Test : Verify Data Quality
do them on df_clean
see [Data Quality](../../../../../journal/trainData/dataQuality.md)
- Are all features stationary
- Does it preserve local memory?
- Are features redundant with each other?
- Does it actually relate to the label?

# Skip to save Data if all had passed

# Note
- all passed, drop RSI_5 because VIF test exceed 30 + High MI correlation

In [ ]:
# Group I — Rate of Change (passed)
# Group L — Order Flow (passed)
# Group J — ATR Volatility Ratio
# Group K — RSI & Momentum Oscillators
# Group M — Distance to Swing High / Low
# Group N — ICT Pivot Distance (15STR + 45STR)

features_to_test = {
    # Group I - passed
    "ROC_3":        df_clean["ROC_3"],
    "ROC_5":        df_clean["ROC_5"],
    "ROC_10":       df_clean["ROC_10"],
    "MOM_3":        df_clean["MOM_3"],
    "RETURNS_1":    df_clean["RETURNS_1"],
    # Group J - passed
    "ATR_5":        df_clean["ATR_5"],
    "ATR_14":       df_clean["ATR_14"],
    "ATR_RATIO":    df_clean["ATR_RATIO"],
    "ATR_NORM_ROC": df_clean["ATR_NORM_ROC"],
    "RANGE_RATIO":  df_clean["RANGE_RATIO"],
    # Group K - passed
    # "RSI_5":        df_clean["RSI_5"], # drop, drop it
    "RSI_14":       df_clean["RSI_14"],
    "RSI_SLOPE":    df_clean["RSI_SLOPE"],
    "STOCH_K":      df_clean["STOCH_K"],
    "CCI_5":        df_clean["CCI_5"], # borderline but pass at 0.449
    # Group L - passed
    "DELTA_1":      df_clean["DELTA_1"],
    "DELTA_3":      df_clean["DELTA_3"],
    "BUY_RATIO":    df_clean["BUY_RATIO"],
    "VOL_SPIKE":    df_clean["VOL_SPIKE"],
    "DELTA_DIV":    df_clean["DELTA_DIV"],
    # Group M - passed
    "DIST_HIGH_5":  df_clean["DIST_HIGH_5"],
    "DIST_LOW_5":   df_clean["DIST_LOW_5"],
    "DIST_HIGH_10": df_clean["DIST_HIGH_10"],
    "DIST_LOW_10":  df_clean["DIST_LOW_10"],
    "RANGE_POS":    df_clean["RANGE_POS"],
    # Group N — log1p on barsSince (sawtooth, non-stationary by construction)
    "15STR_confirmed":  df_clean["15STR_confirmed"],
    "45STR_confirmed":  df_clean["45STR_confirmed"],
    "barsSince15STR":   np.log1p(df_clean["barsSince15STR"]),
    "barsSince45STR":   np.log1p(df_clean["barsSince45STR"]),
    "DIST_HIGH_15STR":  df_clean["DIST_HIGH_15STR"],
    "DIST_LOW_15STR":   df_clean["DIST_LOW_15STR"],
    "DIST_HIGH_45STR":  df_clean["DIST_HIGH_45STR"],
    "DIST_LOW_45STR":   df_clean["DIST_LOW_45STR"],
    "RANGE_15STR":      df_clean["RANGE_15STR"],
    "RANGE_45STR":      df_clean["RANGE_45STR"],
}

In [ ]:
# Test1 : stationary test
# ADF  rejects unit root     → not non-stationary
# KPSS rejects stationarity  → not trend-stationary
from statsmodels.tsa.stattools import adfuller, kpss
# import pandas as pd
# import numpy as np

def stationarity_report(series: pd.Series, name: str) -> dict:
    """
    ADF  null: series HAS a unit root (non-stationary)
         want: reject null → p < 0.05 → stationary

    KPSS null: series IS stationary
         want: fail to reject → p > 0.05 → stationary
    """
    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(series.dropna(), autolag="AIC")
    kpss_stat, kpss_p, _, kpss_crit    = kpss(series.dropna(), regression="c", nlags="auto")

    verdict = "PASS" if (adf_p < 0.05 and kpss_p > 0.05) else \
              "WARN" if (adf_p < 0.05 or  kpss_p > 0.05) else "FAIL"

    return {
        "feature":   name,
        "adf_p":     round(adf_p,   4),
        "kpss_p":    round(kpss_p,  4),
        "adf_stat":  round(adf_stat, 3),
        "kpss_stat": round(kpss_stat,3),
        "verdict":   verdict,
    }

# one-by-one test
# stationarity_report(df["barsSince15STR"],"barsSince15STR")
# stationarity_report(df["barsSince45STR"],"barsSince45STR")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

for feat in ["barsSince15STR", "barsSince45STR"]:
    raw = df.iloc[:160_839][feat]
    transformed = np.log1p(raw)
    
    print(f"{feat}")
    print(f"  raw:      min={raw.min():.0f}  max={raw.max():.0f}  mean={raw.mean():.1f}  std={raw.std():.1f}")
    print(f"  log1p:    min={transformed.min():.3f}  max={transformed.max():.3f}  mean={transformed.mean():.3f}  std={transformed.std():.3f}")
    print()

In [ ]:
# Group N

# feature	adf_p	kpss_p	adf_stat	kpss_stat	verdict
# 0	15STR_confirmed	0.0	0.0431	-61.503	0.494	WARN
# 1	45STR_confirmed	0.0	0.1000	-70.134	0.065	PASS
# 2	barsSince15STR	0.0	0.0100	-50.246	1.292	WARN
# 3	barsSince45STR	0.0	0.1000	-47.477	0.194	PASS
# 4	DIST_HIGH_15STR	0.0	0.0357	-49.844	0.526	WARN
# 5	DIST_LOW_15STR	0.0	0.1000	-49.247	0.183	PASS
# 6	DIST_HIGH_45STR	0.0	0.0100	-33.960	0.905	WARN
# 7	DIST_LOW_45STR	0.0	0.1000	-32.399	0.307	PASS
# 8	RANGE_15STR	0.0	0.1000	-61.728	0.127	PASS
# 9	RANGE_45STR	0.0	0.0929	-35.956	0.363	PASS

# execute stationary test
# results = [stationarity_report(series, name) for name, series in features_to_test.items()]
# pd.DataFrame(results)

In [ ]:
# feature investigation
# ATR_NORM_ROC, RSI_5, RSI_14, DIST_LOW_5, DIST_LOW_10
# check rolling means

window = 5000
for feat in ["15STR_confirmed","DIST_HIGH_15STR", "DIST_HIGH_45STR"]:
    rm = df[feat].rolling(window).mean()
    print(f"{feat}: Q1={rm.iloc[window:60000].mean():.1f}  "
          f"mid={rm.iloc[60000:120000].mean():.1f}  "
          f"Q4={rm.iloc[160000:].mean():.1f}")



# Regime shift : Issues
```
Outcome A — frequency is stable around global mean:
  → KPSS stat inflated by clustering, not a real shift
```

In [ ]:
# Result was outcome A
# that's why we add rolling 500 windows
# ROLLING_WINDOW = 500

# what features' need rolling z-score
# confirmed : ATR_5, ATR_14, STOCH_K, DIST_HIGH_5, DIST_HIGH_10, RANGE_POS
# testing : 
# noaction : ATR_RATIO, RANGE_RATIO, RSI_SLOPE, CCI_5



# Stationary Test End
```
Rolling Z-score (9 features):
  DELTA_1, DELTA_3, BUY_RATIO
  ATR_5, ATR_14
  STOCH_K
  DIST_HIGH_5, DIST_HIGH_10, RANGE_POS

Global Z-score (16 features):
  Everything else

WARN but no action (confirmed artefacts):
  DELTA_DIV   — binary ACF clustering
  ATR_NORM_ROC, RSI_5, RSI_14  — autocorrelation, stable rolling mean
  DIST_LOW_5, DIST_LOW_10      — rolling min clustering, stable rolling mean
```

In [ ]:
# Test2 : local memory test
# Sample Size Inflates Ljung-Box. Use another method

from statsmodels.tsa.stattools import acf
# import pandas as pd
# import numpy as np

def acf_magnitude_report(series: pd.Series, name: str, max_lag: int = 20):
    """
    Look at actual ACF coefficients, not just significance flags.
    Thresholds:
        |r| > 0.10  → meaningful memory
        |r| > 0.05  → weak but present
        |r| < 0.02  → negligible (likely noise even if p < 0.05)
    """
    acf_vals, confint = acf(series.dropna(), nlags=max_lag,
                            alpha=0.05, fft=True)

    # skip lag 0 (always 1.0)
    lags      = range(1, max_lag + 1)
    acf_lags  = acf_vals[1:]
    ci_lower  = confint[1:, 0] - acf_vals[1:]
    ci_upper  = confint[1:, 1] - acf_vals[1:]

    meaningful = [lag for lag, r in zip(lags, acf_lags) if abs(r) > 0.05]
    strong     = [lag for lag, r in zip(lags, acf_lags) if abs(r) > 0.10]

    print(f"\n{name}")
    print(f"  ACF at lag 1:  {acf_lags[0]:+.4f}")
    print(f"  ACF at lag 3:  {acf_lags[2]:+.4f}")
    print(f"  ACF at lag 5:  {acf_lags[4]:+.4f}")
    print(f"  ACF at lag 10: {acf_lags[9]:+.4f}")
    print(f"  ACF at lag 20: {acf_lags[19]:+.4f}")
    print(f"  Lags with |r| > 0.05 (weak):     {meaningful}")
    print(f"  Lags with |r| > 0.10 (meaningful): {strong}")

    return pd.DataFrame({
        "lag": list(lags),
        "acf": acf_lags.round(4),
    })

# run on all features
for name, series in features_to_test.items():
    acf_magnitude_report(series, name)

# group - N 
# a confirmed swing is a point event, not a persistent state.
# extra check: till lag 50
# if some params don't decay to zero

# Local memory test End
```
All 25 features pass local memory check.

25/25 carry meaningful ACF within LSTM receptive field (lags 1–14)
 3/25 extend beyond lag 20 (ATR_5, ATR_14, RSI_14) — structural, acceptable, handled by 500 rolling standardization
 0/25 require any pipeline change based on ACF results
```

In [ ]:
# Test 3 : Cross correlation test
# Are features redundant with each other ?
import matplotlib.pyplot as plt
import seaborn as sns


def correlation_report(df_features: pd.DataFrame, method: str = "spearman"):
    """
    Spearman preferred over Pearson — does not assume linearity.
    Flag pairs with |r| > 0.85 as redundant.
    """
    corr = df_features.corr(method=method)

    # find high-correlation pairs
    pairs = []
    cols  = corr.columns.tolist()
    for i in range(len(cols)):
        for j in range(i+1, len(cols)):
            r = corr.iloc[i, j]
            if abs(r) > 0.85:
                pairs.append({"feature_a": cols[i],
                              "feature_b": cols[j],
                              "spearman_r": round(r, 3)})

    if pairs:
        print("High correlation pairs (|r| > 0.85) — consider dropping one:")
        print(pd.DataFrame(pairs).to_string(index=False))
    else:
        print("No redundant pairs found.")

    # heatmap - plot
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r",
                center=0, vmin=-1, vmax=1)
    plt.title(f"Feature Correlation ({method.title()})")
    plt.tight_layout()
    plt.savefig("feature_correlation.png", dpi=150)
    plt.show()

    return corr

feature_df = pd.DataFrame({name: series for name, series in features_to_test.items()})
correlation_report(feature_df)

In [ ]:
# Test 3 extra - VIF : variance_inflation_factor
# Further Verification, if we really had to drop a column
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
# import numpy as np

def vif_report(X: np.ndarray, feature_names: list):
    vif = pd.DataFrame({
        "feature": feature_names,
        "VIF": [variance_inflation_factor(X, i) for i in range(X.shape[1])]
    }).sort_values("VIF", ascending=False)
    print(vif.to_string(index=False))
    return vif

# Drop RSI_5

feature_names = [
    "ROC_3", "ROC_5", "ROC_10", "MOM_3", "RETURNS_1",                      # Group I
    "DELTA_1", "DELTA_3", "BUY_RATIO", "VOL_SPIKE", "DELTA_DIV",           # Group L
    "ATR_5", "ATR_14", "ATR_RATIO", "ATR_NORM_ROC", "RANGE_RATIO",         # Group J
    "RSI_14", "RSI_SLOPE", "STOCH_K", "CCI_5",                             # Group K
    "DIST_HIGH_5", "DIST_LOW_5", "DIST_HIGH_10", "DIST_LOW_10", "RANGE_POS", # Group M
    "15STR_confirmed", "45STR_confirmed",                                   # Group N (ternary)
    "barsSince15STR", "barsSince45STR",                                     # Group N (log1p)
    "DIST_HIGH_15STR", "DIST_LOW_15STR",                                    # Group N
    "DIST_HIGH_45STR", "DIST_LOW_45STR",                                    # Group N
    "RANGE_15STR", "RANGE_45STR",                                           # Group N
]

df_vif = df_clean.copy()
df_vif["barsSince15STR"] = np.log1p(df_vif["barsSince15STR"])
df_vif["barsSince45STR"] = np.log1p(df_vif["barsSince45STR"])

X_raw = df_vif[feature_names].dropna().values

# Z-score first — VIF on raw unscaled features is misleading
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# run it
# Group N features have remarkably low VIF. = good new orthogonal information
vif_result = vif_report(X_scaled, feature_names)

# Feature Correlation test End
```
RSI_5 is the only feature above the drop threshold of 30. It is almost entirely explained by the combination of STOCH_K, RANGE_POS, DIST_LOW_10, and RSI_SLOPE — all of which encode the same "price position within recent range" information.
```

In [ ]:
# Test 4 - MI : Mutual information 
# predictive signal
# No rolling

In [ ]:
# Test 4 Verify by tweaking k-NN with
# applied 500 rolling window and
# applied Z-score standardization

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
# import pandas as pd
# import numpy as np

# ── feature lists ─────────────────────────────────────────────────────────────
ROLLING_FEATURES = [
    # Group L
    "DELTA_1", "DELTA_3", "BUY_RATIO",
    # Group J
    "ATR_5", "ATR_14",
    # Group K
    "STOCH_K",
    # Group M
    "DIST_HIGH_5", "DIST_HIGH_10", "RANGE_POS",
    # Group N
    "DIST_HIGH_15STR", "DIST_HIGH_45STR",
]  # 11 features

GLOBAL_FEATURES = [
    # Group I
    "ROC_3", "ROC_5", "ROC_10", "MOM_3", "RETURNS_1",
    # Group J
    "ATR_RATIO", "ATR_NORM_ROC", "RANGE_RATIO",
    # Group K
    "RSI_14", "RSI_SLOPE", "CCI_5",
    # Group L
    "VOL_SPIKE", "DELTA_DIV",
    # Group M
    "DIST_LOW_5", "DIST_LOW_10",
    # Group N
    "15STR_confirmed", "45STR_confirmed",
    "barsSince15STR", "barsSince45STR",
    "DIST_LOW_15STR", "DIST_LOW_45STR",
    "RANGE_15STR", "RANGE_45STR",
]  # 23 features

ALL_FEATURES = GLOBAL_FEATURES + ROLLING_FEATURES   # 34 features

ROLLING_WINDOW = 500

# ── Pre-process: log1p on barsSince before any scaling ───────────────────────
df_mi = df_clean.copy()
df_mi["barsSince15STR"] = np.log1p(df_mi["barsSince15STR"])
df_mi["barsSince45STR"] = np.log1p(df_mi["barsSince45STR"])

# ── Step 1: rolling Z-score ───────────────────────────────────────────────────
for feat in ROLLING_FEATURES:
    roll_mean = df_mi[feat].rolling(ROLLING_WINDOW).mean()
    roll_std  = df_mi[feat].rolling(ROLLING_WINDOW).std().replace(0, np.nan)
    df_mi[feat] = (df_mi[feat] - roll_mean) / roll_std

# ── Step 2: drop warmup rows ──────────────────────────────────────────────────
df_mi = df_mi.iloc[ROLLING_WINDOW:].dropna(subset=ALL_FEATURES + ["label"]).reset_index(drop=True)

# ── Step 3: global Z-score on everything else ─────────────────────────────────
X_raw = df_mi[ALL_FEATURES].values
y     = df_mi["label"].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# ── Step 4: winsorise ─────────────────────────────────────────────────────────
X_scaled = np.clip(X_scaled, -3, 3)

# ── Step 5: MI ────────────────────────────────────────────────────────────────
mi_k3  = mutual_info_classif(X_scaled, y, n_neighbors=3,  random_state=42)
mi_k10 = mutual_info_classif(X_scaled, y, n_neighbors=10, random_state=42)

report = pd.DataFrame({
    "feature": ALL_FEATURES,
    "mi_k3":   mi_k3.round(4),
    "mi_k10":  mi_k10.round(4),
}).sort_values("mi_k3", ascending=False)

print(report.to_string(index=False))

In [ ]:
# Diagnose why those MI who return zero

# print(df_clean["DELTA_3"].describe())
# print(df_clean["BUY_RATIO"].describe())

# # Check for near-constant variance
# print("\nDELTA_3  std:", df_clean["DELTA_3"].std())
# print("BUY_RATIO std:", df_clean["BUY_RATIO"].std())

# # Check distribution shape
# print("\nDELTA_3  nunique:", df_clean["DELTA_3"].nunique())
# print("BUY_RATIO nunique:", df_clean["BUY_RATIO"].nunique())

# # Check NaN count
# print("\nDELTA_3  NaN:", df_clean["DELTA_3"].isna().sum())
# print("BUY_RATIO NaN:", df_clean["BUY_RATIO"].isna().sum())

# Mutual Information test End

In [ ]:
# Final test 5 : spearman on feature that is ambiguous
from scipy.stats import spearmanr

for feat in ["barsSince45STR", "barsSince15STR", "15STR_confirmed", "45STR_confirmed"]:
    r, p = spearmanr(df_mi[feat], df_mi["label"])
    print(f"{feat:20s}  r={r:.4f}  p={p:.4f}")


# Export cleaned data

In [6]:
# drop columns concluded from test result
drop_col = ['15STR_confirmed','45STR_confirmed','barsSince15STR','barsSince45STR','RSI_5']
df_clean.drop(columns=drop_col,inplace=True)
print(df_clean.columns)
df_clean.head()

Index(['timestamp', 'label', 'ROC_3', 'ROC_5', 'ROC_10', 'MOM_3', 'RETURNS_1',
       'DELTA_1', 'BUY_RATIO', 'DELTA_3', 'VOL_SPIKE', 'DELTA_DIV', 'ATR_5',
       'ATR_14', 'ATR_RATIO', 'ATR_NORM_ROC', 'RANGE_RATIO', 'RSI_14',
       'RSI_SLOPE', 'STOCH_K', 'CCI_5', 'DIST_HIGH_5', 'DIST_LOW_5',
       'DIST_HIGH_10', 'DIST_LOW_10', 'RANGE_POS', 'DIST_HIGH_15STR',
       'DIST_LOW_15STR', 'DIST_HIGH_45STR', 'DIST_LOW_45STR', 'RANGE_15STR',
       'RANGE_45STR'],
      dtype='object')


,timestamp,label,ROC_3,ROC_5,ROC_10,MOM_3,RETURNS_1,DELTA_1,BUY_RATIO,DELTA_3,...,DIST_LOW_5,DIST_HIGH_10,DIST_LOW_10,RANGE_POS,DIST_HIGH_15STR,DIST_LOW_15STR,DIST_HIGH_45STR,DIST_LOW_45STR,RANGE_15STR,RANGE_45STR
0,1703922900000,-1.0,-0.001003,-0.001423,-0.002869,-1.138651,-0.000306,-5.157845,0.463912,-9.902054,...,0.443375,4.808903,0.443375,0.084416,5.753657,-0.119747,5.753657,-0.723445,-0.021255,-0.143820
1,1703923200000,1.0,-0.000723,-0.001671,-0.003321,-0.827558,-0.000331,-29.107430,0.357073,-27.175385,...,0.433472,5.091789,0.433472,0.078453,6.187813,0.078043,6.187813,-1.109026,0.012455,-0.218364
2,1703923500000,-1.0,0.000471,-0.000227,-0.003441,0.526516,0.001108,4.437141,0.527086,-29.828133,...,1.921415,3.493012,1.921415,0.354870,4.805296,1.681916,4.805296,0.155372,0.259266,0.031321
3,1703923800000,-1.0,0.000310,-0.000082,-0.002196,0.345665,-0.000467,-10.290833,0.403879,-34.961121,...,1.427506,2.232854,1.427506,0.389991,5.318769,1.600766,5.318769,-0.366561,0.231340,-0.074020
4,1703924100000,-1.0,-0.000861,-0.001497,-0.002755,-0.928440,-0.001500,-40.018299,0.299708,-45.871990,...,0.426685,2.532587,0.426685,0.144186,6.760763,-0.072052,6.760763,-1.973753,-0.010772,-0.412314


In [7]:
# save the current data in ./data/mlData/clean-v3
# save to JSONL for training
out_path = folder_path / "clean" / f"{symbol}-{tf}-v10-cleaned.jsonl"

df_clean.to_json(out_path, orient="records", lines=True)
print(f"final shape : {df_clean.shape}")
print(f"Saved {len(df_clean)} rows to {out_path}")


final shape : (229730, 32)
Saved 229730 rows to /Users/aimliu/Library/CloudStorage/OneDrive-Personal/_CODE/Python/py-candle-science/data/mlData/clean/BTCUSDT-5m-v10-cleaned.jsonl
